In [1]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, classification_report, confusion_matrix, precision_score
import numpy as np
from sklearn.base import clone

In [2]:
df = pd.read_csv('foods_health_scores_allergens.csv')
df

,product_name,brands,categories,ingredients,nutriscore_grade,nova_group,ecoscore_grade,allergens,energy_kcal,fat_100g,...,proteins_100g,salt_100g,sodium_100g,contains_gluten,contains_dairy,contains_nuts,contains_soy,contains_eggs,contains_fish,food_type
0,Sidi Ali,سيدي علي,"en:beverages-and-beverages-preparations, en:be...",OBD1 999 999 1112606 266963207 mb,A,NaN,NOT-APPLICABLE,NaN,0.0,0.0,...,0.0,0.000000,0.000000,False,False,False,False,False,False,Branded/Packaged
1,Perly,Perly,"en:dairies, en:fermented-foods, en:fermented-m...","milk cream, cream, sugar, banana, bacteria",UNKNOWN,3.0,B,"en:banana, en:milk",97.0,3.0,...,8.0,NaN,NaN,False,True,False,False,False,False,Branded/Packaged
2,Sidi Ali,Sidi Ali,"en:beverages-and-beverages-preparations, en:be...","Sodium, Calcium, Magnésium, Potassium, Bicarbo...",A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,...,NaN,0.065000,0.026000,False,False,False,False,False,False,Branded/Packaged
3,Eau minérale naturelle,sidi ali,"en:beverages-and-beverages-preparations, en:be...",100% mineral water,A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,...,NaN,0.065000,0.026000,False,False,False,False,False,False,Branded/Packaged
4,اكوافينا,AQUAFINA,"en:beverages-and-beverages-preparations, en:be...",ouverture et avant le : Voir bouteille. après ...,A,NaN,NOT-APPLICABLE,NaN,0.0,0.0,...,0.0,0.000508,0.000203,False,False,False,False,False,False,Branded/Packaged
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4992,Crème fraîche gastronomique,Président,"en:dairies, en:fermented-foods, en:fermented-m...","_Crème_ (origine France), _ferments lactiques_",D,3.0,A,en:milk,291.0,30.0,...,2.3,0.070000,0.028000,False,True,False,False,False,False,Branded/Packaged
4993,Noix de cajou grillées sans sel,Maître Prunille,"en:plant-based-foods-and-beverages, en:plant-b...",Noix de cajou,B,1.0,E,en:nuts,621.0,47.0,...,21.0,0.020000,0.008000,False,False,True,False,False,False,Branded/Packaged
4994,Cacao puro en polvo desgrasado,Valor,"en:cocoa-and-its-products, en:cocoa-and-chocol...","Cacao desgrasado en polvo, correctores de acid...",C,1.0,F,NaN,375.0,16.0,...,26.0,0.030000,0.012000,False,False,False,False,False,False,Branded/Packaged
4995,Sablés Myrtilles Germes de riz,Gerblé,"en:snacks, en:sweet-snacks, en:biscuits-and-cakes","Farine de blé 58,2%, huile de colza, sucre de ...",A,4.0,C,"en:gluten, en:milk",54.0,2.0,...,0.9,0.050000,0.020000,True,True,False,False,False,False,Branded/Packaged


In [3]:
df["food_type"].value_counts()

food_type
Branded/Packaged    4997
Name: count, dtype: int64

In [4]:
df = df.drop(['product_name', 'brands', 'categories', 'ingredients', 'allergens', 'food_type'], axis = 1)

In [5]:
binary_contains_cols = [
    'contains_gluten',
    'contains_dairy',
    'contains_nuts',
    'contains_soy',
    'contains_eggs',
    'contains_fish'
]

## Model Pipeline Definition

In [6]:
# show numeric and categorical columns
num_cols = ['energy_kcal','fat_100g', 'saturated_fat_100g', 'carbs_100g', 'sugars_100g','fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g']     
cat_cols = ['nutriscore_grade', 'nova_group', 'ecoscore_grade']     

# pipeline for numeric
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

# pipeline for categorical
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

## Function Definitions

In [7]:
# Define thresholds
thresholds = np.arange(0.05, 0.96, 0.05)

# Drop unnecessary columns and perform train-test-validation split
def process(df, y_col: str, cols):
    y = df[y_col].astype(int)
    X = df.drop(columns=[y_col], axis=1)

    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=5831,
        stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval,
        test_size=0.25,
        random_state=5831,
        stratify=y_trainval
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

# Get recalls for each threshold
def get_recalls(thresholds, y_prob, y_val):
    recalls = dict()
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        if precision_score(y_val, y_pred, pos_label=1) > 0.5:
            recalls[t] = recall_score(y_val, y_pred, pos_label = 1)
        
    max_key = max(recalls, key=recalls.get)
    return recalls, max_key

# Evaluate on the test set
def evaluate_on_test_set(pipe, X_test, y_test, max_key):
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= max_key)
    
    recall = recall_score(y_test, y_pred, pos_label=1)
    precision = precision_score(y_test, y_pred, pos_label=1)
    
    cm = confusion_matrix(y_test, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Pred 0", "Pred 1"]
    )

    return precision, recall, cm_df

# Get the DataFrame of model coefficients
def get_coef_df(pipe):
    feature_names = pipe.named_steps["preprocess"].get_feature_names_out()
    coefs = pipe.named_steps["lda"].coef_[0]

    coef_df = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefs,
        "abs_coefficient": np.abs(coefs)
    }).sort_values("abs_coefficient", ascending=False)

    return coef_df

def make_pipe(y_col):
    binary_contains_cols_temp = [c for c in binary_contains_cols if c != y_col]

    preprocess = ColumnTransformer([
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols),
        ("bin", "passthrough", binary_contains_cols_temp)
    ])

    pipe = Pipeline([
        ("preprocess", preprocess),
        ("lda", LinearDiscriminantAnalysis())
    ])
    return pipe

## Analysis

### Gluten

In [8]:
gluten_X_train, gluten_X_val, gluten_X_test, gluten_y_train, gluten_y_val, gluten_y_test = process(df, 'contains_gluten', binary_contains_cols)

baseline_recall_gluten = (gluten_y_test == 1).mean()
print("Baseline recall for gluten:", baseline_recall_gluten)

Baseline recall for gluten: 0.32


In [9]:
pipe = make_pipe("contains_gluten")
pipe.fit(gluten_X_train, gluten_y_train)
gluten_y_prob = pipe.predict_proba(gluten_X_val)[:, 1]

In [10]:
gluten_recalls, gluten_max_key = get_recalls(thresholds, gluten_y_prob, gluten_y_val)
print(gluten_max_key)
print(gluten_recalls[gluten_max_key])

0.1
0.9625


In [11]:
gluten_test_precision, gluten_test_recall, gluten_cm_df = evaluate_on_test_set(pipe, gluten_X_test, gluten_y_test, gluten_max_key)
print("Precision:", gluten_test_precision)
print("Recall:", gluten_test_recall)

Precision: 0.5332167832167832
Recall: 0.953125


In [12]:
print(gluten_cm_df)

          Pred 0  Pred 1
Actual 0     413     267
Actual 1      15     305


In [13]:
gluten_coef_df = get_coef_df(pipe)
print(gluten_coef_df)

                                 feature  coefficient  abs_coefficient
3                        num__carbs_100g     3.494051         3.494051
32                    bin__contains_eggs     2.236157         2.236157
1                          num__fat_100g    -2.181360         2.181360
0                       num__energy_kcal     1.879024         1.879024
4                       num__sugars_100g    -1.723703         1.723703
31                     bin__contains_soy     1.582053         1.582053
15         cat__nutriscore_grade_UNKNOWN    -1.383859         1.383859
27    cat__ecoscore_grade_NOT-APPLICABLE    -1.332080         1.332080
14  cat__nutriscore_grade_NOT-APPLICABLE    -1.123820         1.123820
16                   cat__nova_group_1.0    -0.870267         0.870267
17                   cat__nova_group_2.0     0.712603         0.712603
2                num__saturated_fat_100g    -0.694612         0.694612
29                   bin__contains_dairy     0.512635         0.512635
13    

### Dairy

In [14]:

dairy_X_train, dairy_X_val, dairy_X_test, dairy_y_train, dairy_y_val, dairy_y_test = process(df, 'contains_dairy', binary_contains_cols)

baseline_recall_dairy = (dairy_y_test == 1).mean()
print("Baseline recall for dairy:", baseline_recall_dairy)

Baseline recall for dairy: 0.299


In [15]:
pipe = make_pipe("contains_dairy")
pipe.fit(dairy_X_train, dairy_y_train)
dairy_y_prob = pipe.predict_proba(dairy_X_val)[:, 1]

In [16]:
dairy_recalls, dairy_max_key = get_recalls(thresholds, dairy_y_prob, dairy_y_val)
print(dairy_max_key)
print(dairy_recalls[dairy_max_key])

0.25
0.7785234899328859


In [17]:
dairy_test_precision, dairy_test_recall, dairy_cm_df = evaluate_on_test_set(pipe, dairy_X_test, dairy_y_test, dairy_max_key)
print("Precision for dairy:", dairy_test_precision)
print("Recall for dairy:", dairy_test_recall)

Precision for dairy: 0.5434782608695652
Recall for dairy: 0.7525083612040134


In [18]:
print(dairy_cm_df)

          Pred 0  Pred 1
Actual 0     512     189
Actual 1      74     225


In [19]:
dairy_coef_df = get_coef_df(pipe)
print(dairy_coef_df)

                                 feature  coefficient  abs_coefficient
27    cat__ecoscore_grade_NOT-APPLICABLE    -1.817303         1.817303
2                num__saturated_fat_100g     1.678640         1.678640
32                    bin__contains_eggs     1.566610         1.566610
30                    bin__contains_nuts     1.169811         1.169811
33                    bin__contains_fish    -1.042394         1.042394
3                        num__carbs_100g    -1.020736         1.020736
24                 cat__ecoscore_grade_D     0.711956         0.711956
29                  bin__contains_gluten     0.696812         0.696812
31                     bin__contains_soy     0.676265         0.676265
21            cat__ecoscore_grade_A-PLUS    -0.656531         0.656531
13               cat__nutriscore_grade_E     0.623337         0.623337
23                 cat__ecoscore_grade_C     0.605410         0.605410
0                       num__energy_kcal    -0.591597         0.591597
9     

### Nuts

In [20]:
nuts_X_train, nuts_X_val, nuts_X_test, nuts_y_train, nuts_y_val, nuts_y_test = process(df, 'contains_nuts', binary_contains_cols)

baseline_recall_nuts = (nuts_y_test == 1).mean()
print("Baseline recall for nuts:", baseline_recall_nuts)

Baseline recall for nuts: 0.118


In [21]:
pipe = make_pipe("contains_nuts")
pipe.fit(nuts_X_train, nuts_y_train)
nuts_y_prob = pipe.predict_proba(nuts_X_val)[:, 1]

In [22]:
nuts_recalls, nuts_max_key = get_recalls(thresholds, nuts_y_prob, nuts_y_val)
print(nuts_max_key)
print(nuts_recalls[nuts_max_key])

0.45
0.4491525423728814


In [23]:
nuts_test_precision, nuts_test_recall, nuts_cm_df = evaluate_on_test_set(pipe, nuts_X_test, nuts_y_test, nuts_max_key)
print("Precision for nuts:", nuts_test_precision)
print("Recall for nuts:", nuts_test_recall)

Precision for nuts: 0.584070796460177
Recall for nuts: 0.559322033898305


In [24]:
print(nuts_cm_df)

          Pred 0  Pred 1
Actual 0     835      47
Actual 1      52      66


In [25]:
nuts_coef_df = get_coef_df(pipe)
print(nuts_coef_df)

                                 feature  coefficient  abs_coefficient
8                       num__sodium_100g   174.569787       174.569787
7                         num__salt_100g  -174.523798       174.523798
17                   cat__nova_group_2.0    -2.931417         2.931417
33                    bin__contains_fish    -2.090698         2.090698
4                       num__sugars_100g     1.471886         1.471886
1                          num__fat_100g     1.288441         1.288441
15         cat__nutriscore_grade_UNKNOWN    -1.130553         1.130553
2                num__saturated_fat_100g    -1.076868         1.076868
6                     num__proteins_100g     0.821363         0.821363
30                   bin__contains_dairy     0.765481         0.765481
3                        num__carbs_100g    -0.749338         0.749338
31                     bin__contains_soy     0.725813         0.725813
14  cat__nutriscore_grade_NOT-APPLICABLE     0.577402         0.577402
0     

### Soy

In [26]:
soy_X_train, soy_X_val, soy_X_test, soy_y_train, soy_y_val, soy_y_test = process(df, 'contains_soy', binary_contains_cols)

baseline_recall_soy = (soy_y_test == 1).mean()
print("Baseline recall for soy:", baseline_recall_soy)

Baseline recall for soy: 0.17


In [27]:
pipe = make_pipe("contains_soy")
pipe.fit(soy_X_train, soy_y_train)
soy_y_prob = pipe.predict_proba(soy_X_val)[:, 1]

In [28]:
soy_recalls, soy_max_key = get_recalls(thresholds, soy_y_prob, soy_y_val)
print(soy_max_key)
print(soy_recalls[soy_max_key])

0.35000000000000003
0.4260355029585799


In [29]:
soy_test_precision, soy_test_recall, soy_cm_df = evaluate_on_test_set(pipe, soy_X_test, soy_y_test, soy_max_key)
print("Precision for soy:", soy_test_precision)
print("Recall for soy:", soy_test_recall)

Precision for soy: 0.5594405594405595
Recall for soy: 0.47058823529411764


In [30]:
print(soy_cm_df)

          Pred 0  Pred 1
Actual 0     767      63
Actual 1      90      80


In [31]:
soy_coef_df = get_coef_df(pipe)
print(soy_coef_df)

                                 feature  coefficient  abs_coefficient
17                   cat__nova_group_2.0    -2.322700         2.322700
29                  bin__contains_gluten     1.617979         1.617979
3                        num__carbs_100g    -1.553378         1.553378
4                       num__sugars_100g     1.154574         1.154574
21            cat__ecoscore_grade_A-PLUS     1.147391         1.147391
30                   bin__contains_dairy     1.042744         1.042744
19                   cat__nova_group_4.0     0.788883         0.788883
13               cat__nutriscore_grade_E     0.681318         0.681318
31                    bin__contains_nuts     0.622405         0.622405
26                 cat__ecoscore_grade_F     0.608796         0.608796
1                          num__fat_100g     0.566943         0.566943
12               cat__nutriscore_grade_D    -0.510943         0.510943
25                 cat__ecoscore_grade_E     0.489661         0.489661
16    

### Eggs

In [32]:
eggs_X_train, eggs_X_val, eggs_X_test, eggs_y_train, eggs_y_val, eggs_y_test = process(df, 'contains_eggs', binary_contains_cols)

baseline_recall_eggs = (eggs_y_test == 1).mean()
print("Baseline recall for eggs:", baseline_recall_eggs)

Baseline recall for eggs: 0.064


In [33]:
pipe = make_pipe("contains_eggs")
pipe.fit(eggs_X_train, eggs_y_train)
eggs_y_prob = pipe.predict_proba(eggs_X_val)[:, 1]

In [34]:
eggs_recalls, eggs_max_key = get_recalls(thresholds, eggs_y_prob, eggs_y_val)
print(eggs_max_key)
print(eggs_recalls[eggs_max_key])

0.55
0.171875


/opt/anaconda3/envs/dmsp26/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/dmsp26/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/dmsp26/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [35]:
eggs_test_precision, eggs_test_recall, eggs_cm_df = evaluate_on_test_set(pipe, eggs_X_test, eggs_y_test, eggs_max_key)
print("Precision for eggs:", eggs_test_precision)
print("Recall for eggs:", eggs_test_recall)

Precision for eggs: 0.42857142857142855
Recall for eggs: 0.1875


In [36]:
print(eggs_cm_df)

          Pred 0  Pred 1
Actual 0     920      16
Actual 1      52      12


In [37]:
eggs_coef_df = get_coef_df(pipe)
print(eggs_coef_df)

                                 feature  coefficient  abs_coefficient
33                    bin__contains_fish     2.987762         2.987762
29                  bin__contains_gluten     1.986292         1.986292
1                          num__fat_100g     1.945265         1.945265
2                num__saturated_fat_100g    -1.726400         1.726400
30                   bin__contains_dairy     1.530321         1.530321
17                   cat__nova_group_2.0    -1.360345         1.360345
31                    bin__contains_nuts    -1.167793         1.167793
23                 cat__ecoscore_grade_C     1.105511         1.105511
25                 cat__ecoscore_grade_E    -0.896455         0.896455
10               cat__nutriscore_grade_B    -0.868515         0.868515
13               cat__nutriscore_grade_E     0.835011         0.835011
26                 cat__ecoscore_grade_F    -0.705225         0.705225
14  cat__nutriscore_grade_NOT-APPLICABLE    -0.548154         0.548154
6     

### Fish

In [38]:
fish_X_train, fish_X_val, fish_X_test, fish_y_train, fish_y_val, fish_y_test = process(df, 'contains_fish', binary_contains_cols)

baseline_recall_fish = (fish_y_test == 1).mean()
print("Baseline recall for fish:", baseline_recall_fish)

Baseline recall for fish: 0.022


In [39]:
pipe = make_pipe("contains_fish")
pipe.fit(fish_X_train, fish_y_train)
fish_y_prob = pipe.predict_proba(fish_X_val)[:, 1]

In [40]:
fish_recalls, fish_max_key = get_recalls(thresholds, fish_y_prob, fish_y_val)
print(fish_max_key)
print(fish_recalls[fish_max_key])

0.5
0.6086956521739131


In [41]:
fish_test_precision, fish_test_recall, fish_cm_df = evaluate_on_test_set(pipe, fish_X_test, fish_y_test, fish_max_key)
print("Precision for fish:", fish_test_precision)
print("Recall for fish:", fish_test_recall)

Precision for fish: 0.3333333333333333
Recall for fish: 0.5454545454545454


In [42]:
print(fish_cm_df)

          Pred 0  Pred 1
Actual 0     954      24
Actual 1      10      12


In [43]:
fish_coef_df = get_coef_df(pipe)
print(fish_coef_df)

                                 feature  coefficient  abs_coefficient
26                 cat__ecoscore_grade_F     5.471124         5.471124
25                 cat__ecoscore_grade_E     4.421496         4.421496
33                    bin__contains_eggs     3.345542         3.345542
21            cat__ecoscore_grade_A-PLUS    -2.233231         2.233231
16                   cat__nova_group_1.0    -2.124392         2.124392
0                       num__energy_kcal    -2.089373         2.089373
14  cat__nutriscore_grade_NOT-APPLICABLE    -1.958203         1.958203
17                   cat__nova_group_2.0     1.920342         1.920342
29                  bin__contains_gluten     1.854886         1.854886
6                     num__proteins_100g     1.742714         1.742714
18                   cat__nova_group_3.0     1.683307         1.683307
31                    bin__contains_nuts    -1.605143         1.605143
20                 cat__ecoscore_grade_A    -1.453430         1.453430
1     

We observe that our association analysis model has very low precision and recall for every variable except gluten. This could be partly due to a lack of samples containing the other allergens. We intend to tune our model hyperparameters to obtain sounder results since we are especially concerned about low recall because false negatives are more problematic than false positives when determining whether or not something contains an allergen.